# Task 01 — Environment and access discovery

**Purpose:** record your local environment, check Hugging Face and
Qualcomm AI Hub access, and discover which devices, chipsets, and
runtimes you can reach today.

**Lesson:** open [`docs/tasks/01-orientation-and-stack.html`](../docs/tasks/01-orientation-and-stack.html)
first. Read it, then run this notebook.

**Environment note:** run from the project root with the `uv` environment
(`uv run jupyter lab`). Python 3.11. No model download happens here.
Nothing in this notebook prints a secret.

**How to run:** top to bottom. One cell is marked **PARAMETER** — you
will change it and rerun it once. Finish with the final summary cell.
It writes the evidence files to `results/`.

In [ ]:
# Setup: paths and shared state. Run this first.
import json
import os
import platform
import re
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

# Find the project root (the folder that contains progress/state.json).
_here = Path.cwd()
PROJECT_ROOT = next(
    (p for p in [_here, *_here.parents] if (p / "progress" / "state.json").exists()),
    None,
)
assert PROJECT_ROOT is not None, (
    "Project root not found. Start JupyterLab from the project folder: uv run jupyter lab"
)

RESULTS_DIR = PROJECT_ROOT / "results"
SECRETS_FILE = PROJECT_ROOT / ".ai-local" / "secrets" / "qai-hub.env"

# Everything discovered in this notebook accumulates here,
# then the final cell sanitizes and saves it.
DISCOVERY = {"created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}
CHECKS = {}

print("project root:", PROJECT_ROOT)
print("results dir :", RESULTS_DIR)

## Part 1 — Secrets check (nothing is printed)

The next cell looks at `.ai-local/secrets/qai-hub.env`. It reports only
which variable **names** exist and whether each has a value. The values
themselves go into a private redaction list, so if one ever leaks into a
command output later, it is replaced by `[REDACTED]` before saving.

In [ ]:
# Secrets: presence only. Never print values.
_SECRET_VALUES = []          # used by the sanitizer, never displayed
secret_status = {}

if SECRETS_FILE.exists():
    for line in SECRETS_FILE.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        value = value.strip().strip('"').strip("'")
        secret_status[key.strip()] = bool(value)
        if len(value) >= 8:
            _SECRET_VALUES.append(value)
else:
    print("NOTE: secrets file not found at .ai-local/secrets/qai-hub.env")

def sanitize(text: str) -> str:
    """Remove secret values and token-like assignments from text."""
    for v in _SECRET_VALUES:
        text = text.replace(v, "[REDACTED]")
    text = re.sub(r"(?i)((?:api[_-]?token|api[_-]?key|authorization|bearer)\s*[=:]\s*)\S+",
                  r"\1[REDACTED]", text)
    return text

CHECKS["secrets_file_present"] = SECRETS_FILE.exists()
CHECKS["api_token_present"] = any(secret_status.values())
DISCOVERY["secret_variables"] = secret_status  # names + true/false only

print("secrets file present:", CHECKS["secrets_file_present"])
for k, has_value in secret_status.items():
    print(f"  {k}: value present = {has_value}")

## Part 2 — Local environment inventory

Records OS, architecture, Python, disk, and key package versions.
**Inspect:** Python must be 3.11.x. Free disk should be above 30 GB.

In [ ]:
# Local inventory -> saved by the final cell, and also right here.
disk = shutil.disk_usage(PROJECT_ROOT)

def pkg_version(name):
    from importlib.metadata import version, PackageNotFoundError
    try:
        return version(name)
    except PackageNotFoundError:
        return None

inventory = {
    "recorded_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "os": platform.system(),
    "os_version": platform.mac_ver()[0] or platform.release(),
    "architecture": platform.machine(),
    "python_version": platform.python_version(),
    "python_executable_in_venv": ".venv" in sys.executable,
    "disk_free_gb": round(disk.free / 1e9, 1),
    "disk_total_gb": round(disk.total / 1e9, 1),
    "packages": {
        name: pkg_version(name)
        for name in [
            "torch", "transformers", "onnx", "onnxruntime",
            "numpy", "jupyterlab", "qai-hub-models-cli", "qai-hub",
        ]
    },
}

RESULTS_DIR.mkdir(exist_ok=True)
inv_path = RESULTS_DIR / "01_environment_inventory.json"
inv_path.write_text(json.dumps(inventory, indent=2) + "\n")

CHECKS["python_is_311"] = platform.python_version().startswith("3.11.")
CHECKS["disk_free_over_30gb"] = inventory["disk_free_gb"] > 30
CHECKS["inventory_saved"] = inv_path.exists()

print(json.dumps(inventory, indent=2))
print("\nsaved ->", inv_path.relative_to(PROJECT_ROOT))

## Part 3 — Access checks

Two checks, no downloads:

1. **Hugging Face** — can this machine reach the Hub? A token is optional
   because `Qwen/Qwen3-0.6B` is public.
2. **Qualcomm AI Hub client** — is an API token configured for the CLI?
   The real proof comes from the discovery commands in Part 4.

In [ ]:
# Hugging Face access check. No token is printed. No download.
hf = {"reachable": None, "authenticated": None, "note": None}
try:
    from huggingface_hub import HfApi
    api = HfApi()
    info = api.model_info("Qwen/Qwen3-0.6B")   # public metadata call
    hf["reachable"] = True
    hf["note"] = f"model_info ok, likes={info.likes}"
    try:
        who = api.whoami()
        hf["authenticated"] = True
        hf["note"] += ", token works"
    except Exception:
        hf["authenticated"] = False
        hf["note"] += ", anonymous (fine for public models)"
except Exception as e:
    hf["reachable"] = False
    hf["note"] = f"{type(e).__name__}: {str(e)[:200]}"

DISCOVERY["huggingface"] = hf
CHECKS["hf_hub_reachable"] = hf["reachable"] is True
print(json.dumps(hf, indent=2))

In [ ]:
# Qualcomm AI Hub client configuration check. Presence only.
qai_cfg = Path.home() / ".qai_hub" / "client.ini"
DISCOVERY["qai_hub_client_config_present"] = qai_cfg.exists()
CHECKS["qai_hub_configured"] = qai_cfg.exists() or CHECKS["api_token_present"]

print("~/.qai_hub/client.ini present:", qai_cfg.exists())
if not qai_cfg.exists():
    print("Token exists in .ai-local/secrets/qai-hub.env:", CHECKS["api_token_present"])
    print("If discovery commands in Part 4 fail with an auth error,")
    print("ask Claude: Help me with the current task.")

## Part 4 — Qualcomm discovery through the CLI

The helper below runs `qai-hub-models` commands, sanitizes the output,
and stores it in `DISCOVERY`. We record `--help` first: command syntax
comes from the installed version, not from memory.

**Inspect:** in the `devices` output, find at least one device whose
chipset looks like a Snapdragon platform that runs LLM bundles.

In [ ]:
# CLI helper. Every output is sanitized before storing or printing.
def run_cli(*args, timeout=120):
    cmd = ["qai-hub-models", *args]
    try:
        r = subprocess.run(cmd, capture_output=True, text=True,
                           timeout=timeout, cwd=PROJECT_ROOT)
        out = sanitize((r.stdout or "") + (("\n" + r.stderr) if r.stderr else ""))
        return {"command": " ".join(cmd), "returncode": r.returncode,
                "output": out.strip()}
    except FileNotFoundError:
        return {"command": " ".join(cmd), "returncode": -1,
                "output": "qai-hub-models not found. Run: uv pip install -r requirements-core.txt"}
    except subprocess.TimeoutExpired:
        return {"command": " ".join(cmd), "returncode": -1,
                "output": f"timed out after {timeout}s"}

help_result = run_cli("--help")
DISCOVERY["cli_help"] = help_result
CHECKS["cli_available"] = help_result["returncode"] == 0
print(help_result["output"][:3000])

In [ ]:
# Discovery: devices, chipsets, runtimes.
for sub in ["devices", "chipsets", "runtimes"]:
    res = run_cli(sub)
    DISCOVERY[sub] = res
    ok = res["returncode"] == 0
    CHECKS[f"discovery_{sub}"] = ok
    print(f"=== qai-hub-models {sub} -> {'ok' if ok else 'FAILED'} ===")
    print(res["output"][:2500])
    print()

In [ ]:
# Model-specific discovery for Qwen3-0.6B.
# These may need auth or different arguments in this CLI version.
# A failure here is information, not a crash: read the recorded output.
for sub in ["info", "perf"]:
    res = run_cli(sub, "Qwen3-0.6B")
    DISCOVERY[f"model_{sub}"] = res
    ok = res["returncode"] == 0
    CHECKS[f"model_{sub}"] = ok
    print(f"=== qai-hub-models {sub} Qwen3-0.6B -> {'ok' if ok else 'FAILED'} ===")
    print(res["output"][:2500])
    print()

## PARAMETER — device filter comparison

Change `DEVICE_FILTER` below, rerun the cell, and compare the two lists.
For example: `"Snapdragon"` first, then `"Galaxy"`. Each run is recorded,
so run the cell at least twice with different values.

In [ ]:
# PARAMETER: change this value and rerun this cell (at least 2 values).
DEVICE_FILTER = "Snapdragon"

_devices_text = DISCOVERY.get("devices", {}).get("output", "")
matches = [ln for ln in _devices_text.splitlines()
           if DEVICE_FILTER.lower() in ln.lower()]

FILTER_RUNS = globals().get("FILTER_RUNS", {})
FILTER_RUNS[DEVICE_FILTER] = {"match_count": len(matches), "matches": matches[:30]}
DISCOVERY["device_filter_runs"] = FILTER_RUNS
CHECKS["filter_two_values"] = len(FILTER_RUNS) >= 2

print(f"filter {DEVICE_FILTER!r}: {len(matches)} matching device lines")
for ln in matches[:30]:
    print("  ", ln)
print("\nfilters run so far:", list(FILTER_RUNS.keys()))

## Final summary — writes the evidence files

Run this last. It saves `results/01_qualcomm_discovery.json`, scans both
result files for secret leaks, and prints every check. `PASS` lines are
good. `WARN` lines are things to mention when you ask for review.

In [ ]:
# Final summary: save, scan, report.
disc_path = RESULTS_DIR / "01_qualcomm_discovery.json"
disc_path.write_text(sanitize(json.dumps(DISCOVERY, indent=2)) + "\n")

# Secret scan of everything we saved.
leaked = []
for p in [inv_path, disc_path]:
    text = p.read_text()
    if any(v in text for v in _SECRET_VALUES):
        leaked.append(str(p))
CHECKS["no_secrets_in_results"] = not leaked
CHECKS["discovery_saved"] = disc_path.exists()

summary = {
    "task": 1,
    "finished_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "checks": CHECKS,
    "artifacts": [
        str(inv_path.relative_to(PROJECT_ROOT)),
        str(disc_path.relative_to(PROJECT_ROOT)),
    ],
    "device_filters_tried": list(DISCOVERY.get("device_filter_runs", {}).keys()),
}
sum_path = RESULTS_DIR / "01_summary.json"
sum_path.write_text(json.dumps(summary, indent=2) + "\n")

print("=== Task 01 checks ===")
for name, ok in CHECKS.items():
    print(f"  {'PASS' if ok else 'WARN'}  {name}")
print("\nsaved files:")
for a in summary["artifacts"] + [str(sum_path.relative_to(PROJECT_ROOT))]:
    print("  ", a)
if leaked:
    print("\nSTOP: possible secret found in:", leaked)
    print("Do not commit. Tell Claude before saving the notebook.")
else:
    print("\nNo secrets found in saved results.")
print("\nNext: save this notebook, then say: Review my current task.")